# LLM-Augmented Cincinnati, OH / Northern KY Gas Price Prediction Model
### Regional Retail Gasoline Forecasting using LLM Event Sentiment, Mississippi Downriver Logistics & Dual-State Gas Tax Differential (OH: $3.45/gal, KY: $3.325/gal)

This notebook focuses specifically on predicting **retail unleaded gasoline prices across the Cincinnati, OH / Northern Kentucky metropolitan area** (Hamilton County OH & Boone/Kenton/Campbell Counties KY).

### Why Cincinnati, OH & Northern KY are Unique:
1. **Dual-State Gas Tax & Price Differential:** Ohio state motor fuel tax ($0.385/gal) vs Kentucky state motor fuel tax ($0.260/gal) creates a persistent **~$0.125/gal retail price gap** ($3.450/gal OH base vs $3.325/gal KY base).
2. **Marathon Catlettsburg Refinery Proximity:** The 291,000 bpd Catlettsburg KY refinery serves as the primary regional refining benchmark for the Ohio Valley.
3. **Mississippi & Lower Ohio River Downriver Barge Logistics:** Refined fuel barges moving north from Gulf Coast refining hubs enter the Ohio River at the **Cairo, IL confluence** (Mile 981 on Lower Mississippi / Mile 0 on Ohio River). Autumn low-water drought crises (e.g., Memphis/Cairo gage drops) enforce -40% barge payload draft limits, surging spot freight rates +300% and expanding Cincinnati rack margins (+14.5¢/gal).
4. **Ohio River Lock Logistics (Markland Locks & Dam):** Winter ice jams and lock maintenance near Cincinnati choke barge throughput and force reliance on higher-cost rail transport.
5. **Cross-River Consumer Arbitrage:** Commuters frequently cross the Ohio River bridges to fuel up in Northern Kentucky to save ~$0.125/gal.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')

from src.cincinnati_regional import fetch_cincinnati_market_data, get_cincinnati_regional_events
from src.event_analyzer import process_event_dataset, extract_event_features_llm
from src.feature_engineering import create_feature_matrix, prepare_chronological_splits
from src.models import train_and_compare_models

sns.set_theme(style="whitegrid")
print("Cincinnati regional modules successfully loaded!")

In [ ]:
market_df = fetch_cincinnati_market_data(start_date="2022-01-01", live_oh_price=3.450, live_ky_price=3.325)
raw_events_df = get_cincinnati_regional_events()

print(f"Market Trading Days: {len(market_df)}")
print(f"Latest Ohio Pump Price (OH):     ${market_df['cincinnati_oh_retail_gasoline'].iloc[-1]:.3f}/gal")
print(f"Latest Kentucky Pump Price (KY): ${market_df['cincinnati_ky_retail_gasoline'].iloc[-1]:.3f}/gal")
print(f"Cross-River Tax & Price Gap:    ${market_df['oh_ky_tax_spread'].iloc[-1]:.3f}/gal")
display(market_df.head())
display(raw_events_df.head())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(market_df['date'], market_df['cincinnati_oh_retail_gasoline'], label='Cincinnati, OH Retail ($3.45 base / 38.5¢ tax)', color='tab:red', linewidth=2.5)
plt.plot(market_df['date'], market_df['cincinnati_ky_retail_gasoline'], label='Northern Kentucky Retail ($3.325 base / 26.0¢ tax)', color='tab:blue', linewidth=2.5)
plt.plot(market_df['date'], market_df['gasoline_rbob'], label='Wholesale RBOB Futures ($/gal)', color='tab:green', linestyle='--')

plt.title('Cincinnati Metro Cross-River Gas Prices: Ohio vs Northern Kentucky Retail', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('$/Gallon')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
scored_events_df = process_event_dataset(raw_events_df, use_llm_api=False)
feature_df = create_feature_matrix(market_df, scored_events_df, forecast_horizon=5, decay_half_life_days=4.0)
splits = prepare_chronological_splits(feature_df, train_ratio=0.8, forecast_horizon=5)
results = train_and_compare_models(splits, model_type='ridge')
display(pd.DataFrame([results['metrics_quant'], results['metrics_hybrid']], index=['Baseline (Quant Only)', 'Cincinnati Hybrid (Quant + LLM Events)']))